In [1]:
import requests
import csv
import json
import pandas as pd
from IPython.display import display

In [2]:
SERVICE_KEY = '79eA4Jw2odSc97bzYRMnACC2TlJMPETbQmcHQaZaabL7xSMIXQzMcM8Q2xNuWGYbDuHWdL6/eTaSLsIWmgdr/A=='

IOS = 'ios'
AND = 'android'
WEB = 'web'
ETC = 'etc'

MOBILE_OS = [IOS, AND, WEB, ETC]
INPUT_FILENAME = "경기북부관광지.csv"
OUTPUT_FILENAME = "detailCommon2.csv"

fieldnames = ['contentid','contenttypeid','title','createdtime','modifiedtime','tel','telname','homepage','firstimage','firstimage2','cpyrhtDivCd','areacode','sigungucode','lDongRegnCd','lDongSignguCd','lclsSystm1','lclsSystm2','lclsSystm3','cat1','cat2','cat3','addr1','addr2','zipcode','mapx','mapy','mlevel','overview']

In [3]:
def fetch_and_convert_to_csv(content_ids, save_each_step=True):
    """
    주어진 content_ids 리스트를 사용하여 API를 호출하고,
    중간에 오류가 발생하더라도 현재까지 수집된 데이터를 DataFrame으로 반환합니다.
    """

    print(f"총 {len(content_ids)}개의 contentId에 대해 API를 호출합니다.")

    all_items = []
    failed_ids = []

    try:
        for idx, contentId in enumerate(content_ids, start=1):
            if not contentId:
                continue

            print(f"\n[{idx}/{len(content_ids)}] -> contentId: {contentId} 호출 중...")

            API_URL = (
                f"https://apis.data.go.kr/B551011/KorService2/detailCommon2"
                f"?MobileOS={MOBILE_OS[3]}"
                f"&MobileApp=Stay-Sync"
                f"&_type=json"
                f"&contentId={contentId}"
                f"&serviceKey={SERVICE_KEY}"
            )

            try:
                response = requests.get(API_URL, timeout=10)
                response.raise_for_status()

                data = response.json()

                items = data.get("response", {}) \
                            .get("body", {}) \
                            .get("items", {}) \
                            .get("item", [])

                if items:
                    # item이 dict 하나로 오는 경우 대비
                    if isinstance(items, dict):
                        items = [items]

                    all_items.extend(items)
                    print(f"   -> 성공적으로 {len(items)}개 항목을 가져왔습니다.")
                else:
                    print(f"   -> contentId {contentId}에 대한 항목이 없습니다.")

                # 매 호출마다 임시 CSV 저장
                if save_each_step and all_items:
                    temp_df = pd.DataFrame(all_items)
                    temp_df = temp_df.reindex(columns=fieldnames)
                    temp_df.to_csv(
                        "detailCommon2_temp_checkpoint.csv",
                        index=False,
                        encoding="utf-8-sig"
                    )

            except requests.exceptions.RequestException as e:
                print(f"⚠️ 요청 오류 발생: contentId={contentId}, error={e}")
                failed_ids.append(contentId)
                continue

            except json.JSONDecodeError as e:
                print(f"⚠️ JSON 변환 오류 발생: contentId={contentId}, error={e}")
                failed_ids.append(contentId)
                continue

            except Exception as e:
                print(f"🚨 처리 중 오류 발생: contentId={contentId}, error={e}")
                failed_ids.append(contentId)
                continue

    except KeyboardInterrupt:
        print("\n⛔ 사용자가 실행을 중단했습니다.")
        print("현재까지 수집된 데이터로 DataFrame을 생성합니다.")

    finally:
        if not all_items:
            print("\n❌ 현재까지 수집된 유효 데이터가 없습니다.")
            return pd.DataFrame(), failed_ids

        df = pd.DataFrame(all_items)
        df = df.reindex(columns=fieldnames)

        df.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")

        print(f"\n✅ 현재까지 총 {len(df)}개의 데이터를 수집했습니다.")
        print(f"🎉 CSV 파일 '{OUTPUT_FILENAME}' 저장이 완료되었습니다.")

        if failed_ids:
            print(f"\n⚠️ 실패한 contentId 수: {len(failed_ids)}개")
            print(failed_ids[:20])

        return df, failed_ids

In [4]:
df_input = pd.read_csv(INPUT_FILENAME)

if "contentid" not in df_input.columns:
    print(f"🚨 오류: 입력 파일 '{INPUT_FILENAME}'에 'contentid' 컬럼이 없습니다.")
else:
    content_ids = df_input["contentid"].dropna().tolist()

    result_df, failed_ids = fetch_and_convert_to_csv(content_ids)

    display(result_df)

총 3173개의 contentId에 대해 API를 호출합니다.

[1/3173] -> contentId: 3482667 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[2/3173] -> contentId: 129194 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[3/3173] -> contentId: 2894914 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[4/3173] -> contentId: 2831563 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[5/3173] -> contentId: 125462 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[6/3173] -> contentId: 2846052 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[7/3173] -> contentId: 2901496 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[8/3173] -> contentId: 798518 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[9/3173] -> contentId: 2627867 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[10/3173] -> contentId: 3045767 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[11/3173] -> contentId: 2726655 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[12/3173] -> contentId: 2748296 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[13/3173] -> contentId: 125463 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[14/3173] -> contentId: 2832447 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[15/3173] -> contentId: 2

,contentid,contenttypeid,title,createdtime,modifiedtime,tel,telname,homepage,firstimage,firstimage2,...,cat1,cat2,cat3,addr1,addr2,zipcode,mapx,mapy,mlevel,overview
0,3482667,12,가금철교 문화공원,20250318170121,20250318170732,,,,http://tong.visitkorea.or.kr/cms/resource/68/3...,http://tong.visitkorea.or.kr/cms/resource/68/3...,...,A02,A0202,A02020700,경기도 의정부시 가능동,15-28,11685,127.0455151412,37.7533531645,6,가금철교 문화공원은 경기도 의정부시 가능동 을지대학병원 맞은편에 위치해 있다. 원래...
1,129194,12,가나아트파크,20060807090000,20251030173026,,,"<a href=""http://www.artpark.co.kr/"" target=""_b...",http://tong.visitkorea.or.kr/cms/resource/83/3...,http://tong.visitkorea.or.kr/cms/resource/83/3...,...,A02,A0202,A02020600,경기도 양주시 장흥면 권율로 117,,11520,126.9491636401,37.7250130611,6,"가나아트파크는 1984년 국내 최초 사립미술관인 토탈미술관이 그 전신으로서, 200..."
2,2894914,39,가나안국수,20221103152309,20250418143546,,,"<a href=""http://www.xn--o39a20ar3ebs0asud.kr/""...",http://tong.visitkorea.or.kr/cms/resource/91/2...,http://tong.visitkorea.or.kr/cms/resource/91/2...,...,A05,A0502,A05020100,경기도 고양시 덕양구 행주로15번길 16,,10440,126.8278865647,37.6009169989,6,"경기도 고양시 덕양구 행주로에 위치한 가나안국수는 내산 사골, 닭고기, 사태, 멸치..."
3,2831563,39,가나안장어마을,20220803105028,20250421133912,,,,http://tong.visitkorea.or.kr/cms/resource/54/2...,http://tong.visitkorea.or.kr/cms/resource/54/2...,...,A05,A0502,A05020100,경기도 고양시 일산동구 애니골길 54 (풍동),,10301,126.7921100429,37.6721941761,6,가나안장어마을은 고양시 일산동구 풍동 애니골에 자리 잡고 있다. 가나안장어마을은 애...
4,125462,12,가덕산,20030331090000,20251120135219,,,,http://tong.visitkorea.or.kr/cms/resource/24/1...,http://tong.visitkorea.or.kr/cms/resource/24/1...,...,A01,A0101,A01010400,경기도 가평군 북면 목동리,산 1-1,12403,127.6121190936,37.9405556371,6,"경기 제1봉인 화악산(1,468m)에서 동남쪽으로 뻗어 내린 능선 위에 솟아있는 가..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,125549,12,고양 벽제관지,20020510090000,20260312092314,,,http://www.goyang.go.kr/visitgoyang,https://tong.visitkorea.or.kr/cms/resource/16/...,https://tong.visitkorea.or.kr/cms/resource/16/...,...,,,,경기 고양시 덕양구 고양동,,10274,126.9007848164469,37.70455754969595,6,고양 벽제관지는 조선시대의 객사인 벽제관이 위치했던 장소이다. 1476년(성종 7)...
147,125551,12,고양 서삼릉 [유네스코 세계유산],20020510090000,20251211174642,,,"<a href=""https://royal.khs.go.kr/ROYAL/content...",,,...,A02,A0201,A02010700,경기도 고양시 덕양구 서삼릉길 233-126 (원당동),,10292,126.8669756845,37.6630660206,6,서삼릉(西三陵)은 ‘서쪽에 있는 3기의 능’이라는 뜻으로 경기 고양시에 위치한 조선...
148,125552,12,고양 서오릉 [유네스코 세계유산],20021014090000,20260327092150,,,https://royal.khs.go.kr,https://tong.visitkorea.or.kr/cms/resource/20/...,https://tong.visitkorea.or.kr/cms/resource/20/...,...,,,,경기도 고양시 덕양구 서오릉로 334-32,,10548,126.900766297274,37.6235552310618,6,"서오릉(西五陵)은 ‘서쪽에 있는 5기의 능’이라는 뜻으로 경릉(敬陵), 창릉(昌陵)..."
149,1105623,14,고양 선인장전시관,20101009011521,20250416173305,,,,http://tong.visitkorea.or.kr/cms/resource/38/3...,http://tong.visitkorea.or.kr/cms/resource/38/3...,...,A02,A0206,A02060300,경기도 고양시 일산동구 호수로 731 (장항동),,10400,126.7604631520,37.6645195058,6,고양시 일산 호수 공원 내 서쪽에 있는 고양 선인장 전시관은 고양시 농업기술센터에서...


In [5]:
def retry_failed_and_append(failed_ids):
    """
    실패한 contentId만 다시 호출하고,
    기존 OUTPUT_FILENAME CSV에 이어붙여 저장합니다.
    """

    # 1. 기존 저장 파일 불러오기
    try:
        existing_df = pd.read_csv(OUTPUT_FILENAME)
        print(f"기존 저장 데이터: {len(existing_df)}건")
    except FileNotFoundError:
        existing_df = pd.DataFrame(columns=fieldnames)
        print("기존 CSV 파일이 없어 새로 생성합니다.")

    # 2. 기존 성공 contentid 확인
    if "contentid" in existing_df.columns:
        existing_ids = existing_df["contentid"].dropna().astype(str).unique().tolist()
    else:
        existing_ids = []

    # 3. 실패 ID 중 이미 저장된 ID 제외
    retry_ids = [
        cid for cid in failed_ids
        if str(cid) not in existing_ids
    ]

    print(f"재시도 대상 contentId: {len(retry_ids)}개")

    if not retry_ids:
        print("재시도할 contentId가 없습니다.")
        return existing_df, []

    # 4. 실패 ID만 다시 호출
    retry_df, still_failed_ids = fetch_and_convert_to_csv(
        retry_ids,
        save_each_step=True
    )

    if retry_df.empty:
        print("재시도 결과 새로 수집된 데이터가 없습니다.")
        return existing_df, still_failed_ids

    # 5. 기존 데이터 + 재시도 성공 데이터 합치기
    combined_df = pd.concat([existing_df, retry_df], ignore_index=True)

    # 6. contentid 기준 중복 제거
    if "contentid" in combined_df.columns:
        combined_df["contentid"] = combined_df["contentid"].astype(str)
        combined_df = combined_df.drop_duplicates(subset=["contentid"], keep="first")

    # 7. 컬럼 순서 정리
    combined_df = combined_df.reindex(columns=fieldnames)

    # 8. 최종 저장
    combined_df.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")

    print(f"\n✅ 이어서 저장 완료")
    print(f"최종 저장 데이터: {len(combined_df)}건")
    print(f"아직 실패한 contentId: {len(still_failed_ids)}개")

    return combined_df, still_failed_ids

In [6]:
combined_df, still_failed_ids = retry_failed_and_append(failed_ids)

display(combined_df)

NameError: name 'failed_ids' is not defined